# Error Analysis — Urgency Classifier

**Objetivo:** analisar erros reais do modelo treinado no conjunto de validação (`val.csv`), com foco nos erros mais graves clinicamente — casos verdadeiramente `urgent` que o modelo classificou como `normal`.

**Por que isso importa:** o macro F1 (0.813) e o F1 por classe já mostram que `normal` é a classe mais fraca (0.72), mas não explicam *por quê* o modelo erra, nem mostram exemplos concretos. Esta análise investiga a causa raiz de erros reais, documentados (não hipotéticos), usando o modelo já treinado.

## 1. Carregar o modelo treinado e o conjunto de validação

In [1]:
import sys

sys.path.insert(0, "../src")

import joblib
import pandas as pd

preprocessor = joblib.load("../model_artifacts/preprocessor.joblib")

from app.models.logistic_classifier import LogisticClassifier

classifier = LogisticClassifier()
classifier.load("../model_artifacts/classifier.joblib")

val = pd.read_csv("../data/processed/val.csv")
print(f"Val set: {len(val)} amostras")
val.head(3)

Val set: 1282 amostras


,medical_abstract,urgency_label,source
0,Long-term results of submandibular duct transp...,normal,train
1,Bronchial anomaly of the right upper lobe. Thi...,attention,train
2,Technetium-99m sestamibi myocardial imaging: s...,urgent,train


In [6]:
import sys

print("Python executando:", sys.executable)

Python executando: c:\Users\letic\OneDrive\Área de Trabalho\Tech Challenge 3\Tech-Challenge-3\.venv\Scripts\python.exe


## 2. Gerar previsões em todo o conjunto de validação

In [2]:
val_texts = val["medical_abstract"].tolist()
val_features = preprocessor.transform(val_texts)

val = val.copy()
val["predicted"] = classifier.predict(val_features)
val_probabilities = classifier.predict_proba(val_features)

val.head(3)

,medical_abstract,urgency_label,source,predicted
0,Long-term results of submandibular duct transp...,normal,train,normal
1,Bronchial anomaly of the right upper lobe. Thi...,attention,train,attention
2,Technetium-99m sestamibi myocardial imaging: s...,urgent,train,urgent


## 3. Quantificar os erros mais graves

Foco no pior tipo de erro clinicamente: casos **verdadeiramente `urgent`** que o modelo classificou como **`normal`** — o oposto de um sistema de triagem seguro (atraso de atendimento em uma emergência real).

In [3]:
severe_errors = val[(val["urgency_label"] == "urgent") & (val["predicted"] == "normal")]

total_urgent = (val["urgency_label"] == "urgent").sum()
pct_severe = len(severe_errors) / total_urgent * 100

print(f"Total de erros graves (real=urgent, previsto=normal): {len(severe_errors)}")
print(f"Total de casos urgent reais no val: {total_urgent}")
print(f"Proporção de casos urgent classificados incorretamente como normal: {pct_severe:.1f}%")

Total de erros graves (real=urgent, previsto=normal): 76
Total de casos urgent reais no val: 469
Proporção de casos urgent classificados incorretamente como normal: 16.2%


## 4. Inspecionar exemplos reais

In [4]:
classes = classifier.classes()

for i, (idx, row) in enumerate(severe_errors.head(3).iterrows()):
    print(f"--- Exemplo {i + 1} ---")
    print(f"Texto: {row['medical_abstract'][:200]}...")
    idx_pos = val.index.get_loc(idx)
    probs = val_probabilities[idx_pos]
    for c, p in zip(classes, probs, strict=False):
        print(f"  {c}: {p:.4f}")
    print()

--- Exemplo 1 ---
Texto: The use of cognitive behavior therapy with a normalizing rationale in schizophrenia. Preliminary report. Sixty-four consecutively referred patients with schizophrenia were treated with cognitive-behav...
  attention: 0.1536
  normal: 0.4630
  urgent: 0.3834

--- Exemplo 2 ---
Texto: Ulnar nerve decompression with medial epicondylectomy for neuropathy at the elbow. Ulnar nerve decompression with medial epicondylectomy was performed in 66 elbows between 1966 and 1986 for compressiv...
  attention: 0.2220
  normal: 0.4556
  urgent: 0.3224

--- Exemplo 3 ---
Texto: Angioscopy for intraoperative management of thromboembolectomy. Our experience with angioscopy suggests that direct visualization of the arterial lumen during thromboembolectomy procedures would provi...
  attention: 0.0896
  normal: 0.7814
  urgent: 0.1290



## 5. Investigar a causa: quais palavras pesam contra a classificação correta?

Para o exemplo mais claro, olhamos o coeficiente aprendido pelo `LogisticRegression` para os termos com maior peso TF-IDF no texto — isso revela se palavras específicas estão "puxando" a previsão para a classe errada.

In [5]:
example = severe_errors.iloc[0]
example_text = example["medical_abstract"]

example_features = preprocessor.transform([example_text])
vocab = preprocessor.vectorizer.vocabulary_
feature_names = preprocessor.vectorizer.get_feature_names_out()

row = example_features.toarray()[0]
nonzero = [(feature_names[i], row[i]) for i in range(len(row)) if row[i] > 0]
nonzero.sort(key=lambda x: -x[1])

print("Termos com maior peso TF-IDF neste texto:")
print(f"{'termo':20s} {'tfidf':>8s} {'attention':>12s} {'normal':>10s} {'urgent':>10s}")
for term, weight in nonzero[:10]:
    idx = vocab[term]
    coefs = classifier.classifier.coef_[:, idx]
    print(f"{term:20s} {weight:8.4f} {coefs[0]:12.3f} {coefs[1]:10.3f} {coefs[2]:10.3f}")

Termos com maior peso TF-IDF neste texto:
termo                   tfidf    attention     normal     urgent
rationale              0.3434        0.144     -0.097     -0.048
schizophrenia          0.3349       -0.454      0.147      0.308
cognitive              0.2910       -0.544      0.017      0.527
techniques             0.2148        0.096     -0.008     -0.088
normalizing            0.2132       -0.014      0.040     -0.026
psychotherapy          0.2132       -0.099      0.109     -0.011
emergence              0.1860       -0.023      0.012      0.011
devised                0.1768        0.019      0.038     -0.057
consecutively          0.1733       -0.005     -0.068      0.074
supplemented           0.1733       -0.165      0.274     -0.109


## 6. Conclusão

**Achado quantificado:** 16,2% dos casos verdadeiramente `urgent` no conjunto de validação são classificados incorretamente como `normal` — o tipo de erro mais grave possível para um sistema de triagem.

**Causa provável, com evidência:** a categoria original `nervous system diseases` (mapeada para `urgent`) inclui tanto emergências neurológicas quanto procedimentos ortopédicos/administrativos de rotina — o vocabulário desses textos nem sempre reflete urgência clínica real, mesmo estando corretamente mapeado pela categoria de doença. Termos específicos do texto podem ter coeficiente aprendido favorecendo outras classes, reflet indo associações estatísticas do corpus de treino (artigos científicos), não necessariamente o uso clínico esperado em um contexto real de triagem.

**Limitação documentada:** modelos lineares bag-of-words (TF-IDF + Logistic Regression) não capturam contexto semântico — dependem inteiramente de quais palavras específicas apareceram, e com que frequência, em cada classe durante o treino. Um vocabulário urgente do ponto de vista clínico pode não coincidir com o vocabulário estatisticamente associado à urgência no corpus de treino disponível.